In [ ]:
!pip install --upgrade synthcity
!pip show synthcity

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchtext torchmetrics torchao torchtune torchtuples
!pip install torch==2.4.0 torchvision==0.19.0 torchtext==0.18.0

In [ ]:
!pip install torchtuples
!pip uninstall numpy -y
!pip install --upgrade "numpy==2.0.0" "networkx>=3.2" 

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/datasets/marenghimanuela/dataset-ricoveri/dataset_ricoveri_encode_index.csv',
    sep=';',
    quotechar='"',
    encoding='utf-8',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)

In [10]:
# stdlib
import sys
import warnings

warnings.filterwarnings("ignore")

# synthcity absolute
#import synthcity.logger as log

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

cod_regs = df["COD_REG"].dropna().unique()

train_cod, test_cod = train_test_split(
    cod_regs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_df = df[df["COD_REG"].isin(train_cod)].copy()
test_df  = df[df["COD_REG"].isin(test_cod)].copy()

In [ ]:
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from synthcity.plugins.core.dataloader import TimeSeriesSurvivalDataLoader


class RicoveriTimeSeriesDataloader:
    """
    Prepara i dati di ricovero per synthcity TimeSeriesDataLoader.
    """

    TEMPORAL_CONTINUOUS = [
        "eta",
        "qt_prest_Sum",
    ]

    TEMPORAL_BINARY = [
        "SHOCK",
        "CABG",
        "PTCA",
        "ICD"
    ]

    TEMPORAL_CATEGORICAL = [
        "class_prest",
    ]

    TIME_INDEX_COL = "time_days"

    STATIC_CONTINUOUS = [
        "MCS_score",
    ]

    STATIC_BINARY = [
        "sesso",
    ]

    STATIC_CATEGORICAL = [
        "gruppo",
    ]

    OUTCOME_EVENT_COL = "dec_intra"
    OUTCOME_TIME_COL = "death_time"

    def __init__(
        self,
        seq_len: int = 5,
        step: int = 5,
        max_windows_per_patient: int = 1,
        balance_deceased: bool = True,
        n_patients: int = 2000,
        random_seed: int = 42,
    ):
        self.seq_len = seq_len
        self.step = step
        self.max_windows = max_windows_per_patient
        self.balance_deceased = balance_deceased
        self.n_patients = n_patients
        self.seed = random_seed

        self.scaler_temp = MinMaxScaler()
        self.scaler_static = MinMaxScaler()
        self.encoder_temp = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.encoder_static = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

        self.imputer_temp_cont = SimpleImputer(strategy="median")
        self.imputer_temp_cat = SimpleImputer(strategy="most_frequent")
        self.imputer_static_cont = SimpleImputer(strategy="median")
        self.imputer_static_cat = SimpleImputer(strategy="most_frequent")

    def _select_patients(self, df: pd.DataFrame) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)

        eligible = [
            cod for cod, grp in df.groupby("COD_REG")
            if len(grp) > 2 and (grp["desc_studio_out"] != 2).any()
        ]

        deceased = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 1]
        survived = [c for c in eligible if df[df["COD_REG"] == c]["dec_intra"].max() == 0]

        if self.balance_deceased:
            n_dec = min(len(deceased), self.n_patients // 2)
            n_surv = min(len(survived), self.n_patients - n_dec)
        else:
            n_dec = min(len(deceased), self.n_patients)
            n_surv = 0

        chosen_dec = rng.choice(deceased, size=n_dec, replace=False)
        chosen_surv = rng.choice(survived, size=n_surv, replace=False)
        chosen = np.concatenate([chosen_dec, chosen_surv])

        print(f"Pazienti selezionati → deceduti: {n_dec}, sopravvissuti: {n_surv}, totale: {len(chosen)}")
        return df[df["COD_REG"].isin(chosen)].copy()

    def _preprocess_temporal(self, data: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.TEMPORAL_CONTINUOUS:
            cont = data[self.TEMPORAL_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.fit_transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.fit_transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_temp_cont.transform(cont),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
                cont_scaled = pd.DataFrame(
                    self.scaler_temp.transform(cont_imp),
                    columns=self.TEMPORAL_CONTINUOUS, index=data.index
                )
            data[self.TEMPORAL_CONTINUOUS] = cont_scaled

        if self.TEMPORAL_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            data[self.TEMPORAL_BINARY] = bin_imp.fit_transform(data[self.TEMPORAL_BINARY])
            data[self.TEMPORAL_BINARY] = data[self.TEMPORAL_BINARY].astype(int)

        if self.TEMPORAL_CATEGORICAL:
            cat = data[self.TEMPORAL_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.fit_transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.fit_transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_temp_cat.transform(cat),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
                cat_enc = pd.DataFrame(
                    self.encoder_temp.transform(cat_imp),
                    columns=self.TEMPORAL_CATEGORICAL, index=data.index
                )
            data[self.TEMPORAL_CATEGORICAL] = cat_enc

        return data

    def _preprocess_static(self, static: pd.DataFrame, fit: bool = True) -> pd.DataFrame:

        if self.STATIC_CONTINUOUS:
            cont = static[self.STATIC_CONTINUOUS].copy()
            if fit:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.fit_transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.fit_transform(cont_imp)
            else:
                cont_imp = pd.DataFrame(
                    self.imputer_static_cont.transform(cont),
                    columns=self.STATIC_CONTINUOUS, index=static.index
                )
                static[self.STATIC_CONTINUOUS] = self.scaler_static.transform(cont_imp)

        if self.STATIC_BINARY:
            bin_imp = SimpleImputer(strategy="most_frequent")
            static[self.STATIC_BINARY] = bin_imp.fit_transform(static[self.STATIC_BINARY])
            static[self.STATIC_BINARY] = static[self.STATIC_BINARY].astype(int)

        if self.STATIC_CATEGORICAL:
            cat = static[self.STATIC_CATEGORICAL].copy()
            if fit:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.fit_transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.fit_transform(cat_imp)
            else:
                cat_imp = pd.DataFrame(
                    self.imputer_static_cat.transform(cat),
                    columns=self.STATIC_CATEGORICAL, index=static.index
                )
                static[self.STATIC_CATEGORICAL] = self.encoder_static.transform(cat_imp)

        return static

    def _make_windows(
        self,
        group: pd.DataFrame,
        temporal_cols: List[str],
    ) -> Tuple[List[pd.DataFrame], List[np.ndarray]]:

        windows, times = [], []
        n = len(group)

        indices = range(0, max(1, n - self.seq_len + 1), self.step)
        chosen_indices = list(indices)[: self.max_windows]

        for start in chosen_indices:
            end = start + self.seq_len
            chunk = group.iloc[start:end]

            if len(chunk) < self.seq_len:
                pad_rows = self.seq_len - len(chunk)
                pad = pd.concat([chunk.iloc[[-1]]] * pad_rows, ignore_index=True)
                chunk = pd.concat([chunk, pad], ignore_index=True)

            windows.append(chunk[temporal_cols].reset_index(drop=True))
            times.append(chunk[self.TIME_INDEX_COL].values)

        return windows, times

    def inverse_static(self, static_df: pd.DataFrame) -> pd.DataFrame:
        df = static_df.copy()
    
        # CONTINUE
        if self.STATIC_CONTINUOUS:
            df[self.STATIC_CONTINUOUS] = self.scaler_static.inverse_transform(
                df[self.STATIC_CONTINUOUS]
            )
    
        # CATEGORICHE
        if self.STATIC_CATEGORICAL:
            df[self.STATIC_CATEGORICAL] = self.encoder_static.inverse_transform(
                df[self.STATIC_CATEGORICAL]
            )
    
        return df

    def inverse_temporal(self, windows: List[pd.DataFrame]) -> List[pd.DataFrame]:
        inv_windows = []
    
        for w in windows:
            w = w.copy()
    
            # CONTINUE
            if self.TEMPORAL_CONTINUOUS:
                w[self.TEMPORAL_CONTINUOUS] = self.scaler_temp.inverse_transform(
                    w[self.TEMPORAL_CONTINUOUS]
                )
    
            # CATEGORICHE
            if self.TEMPORAL_CATEGORICAL:
                w[self.TEMPORAL_CATEGORICAL] = self.encoder_temp.inverse_transform(
                    w[self.TEMPORAL_CATEGORICAL]
                )
    
            inv_windows.append(w)
    
        return inv_windows
    
    def load(
        self,
        df: pd.DataFrame,
        fit: bool = True,
    ) -> TimeSeriesSurvivalDataLoader:

        data = self._select_patients(df) if fit else df.copy()
        data = data.sort_values(["COD_REG", "time_step"]).reset_index(drop=True)

        temporal_cols = self.TEMPORAL_CONTINUOUS + self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL
        data = self._preprocess_temporal(data, fit=fit)

        all_windows, all_times, static_rows, T_list, E_list = [], [], [], [], []

        for cod, group in data.groupby("COD_REG"):
            if len(group) < 2:
                continue

            wins, times = self._make_windows(group, temporal_cols)

            for w, t in zip(wins, times):
                w = w.fillna(0)  # ✅ FIX TEMPORAL

                all_windows.append(w)
                all_times.append(t)

                static_row = group[
                    self.STATIC_CONTINUOUS + self.STATIC_BINARY + self.STATIC_CATEGORICAL
                ].iloc[-1]

                static_rows.append(static_row)

                E_list.append(int(group[self.OUTCOME_EVENT_COL].max()))
                T_list.append(float(group[self.OUTCOME_TIME_COL].max()))

        static_df = pd.DataFrame(static_rows).reset_index(drop=True)
        static_df = self._preprocess_static(static_df, fit=fit)

        static_df = static_df.fillna(0)  # ✅ FIX STATIC

        T = np.array(T_list, dtype=float)
        E = np.array(E_list, dtype=float)
        
        # ✅ FIX CRITICO
        T = np.nan_to_num(T, nan=0.0, posinf=0.0, neginf=0.0)
        E = np.nan_to_num(E, nan=0.0)
        
        E = E.astype(int)

        # ✅ DEBUG CHECK (opzionale)
        assert not np.isnan(static_df.values).any(), "NaN nelle static!"
        assert not any(w.isna().any().any() for w in all_windows), "NaN nelle temporal!"

        print(f"Finestre totali: {len(all_windows)}")
        print(f"Pazienti con evento (dec_intra=1): {E.sum()} / {len(E)}")

        loader = TimeSeriesSurvivalDataLoader(
            temporal_data=all_windows,
            observation_times=all_times,
            static_data=static_df,
            T=T,
            E=E,
            static_categorical_columns=self.STATIC_BINARY + self.STATIC_CATEGORICAL,
            temporal_categorical_columns=self.TEMPORAL_BINARY + self.TEMPORAL_CATEGORICAL,
        )

        return loader

In [ ]:
l = RicoveriTimeSeriesDataloader(
      seq_len=5,
      step=5,
      max_windows_per_patient=1,
      balance_deceased=True,   # ← FIX: include anche sopravvissuti
      n_patients=5000,
  )

loader_train = l.load(train_df, fit=True)

In [12]:
from synthcity.utils.serialization import save, load
from synthcity.utils.serialization import save_to_file, load_from_file

synthetic = load_from_file('/kaggle/input/datasets/marenghimanuela/synthetic-dataset-dataframe/timegan_10000pz_synthetic.csv')

In [13]:
top_cod = synthetic['seq_id'].unique()[:5000]
synthetic = synthetic[synthetic['seq_id'].isin(top_cod)]

In [14]:
synthetic = synthetic.sort_values(by=['seq_id', 'seq_time_id'])

In [15]:
real = loader_train.dataframe()

In [ ]:
pip install scikit-survival

In [18]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

X_real = real.drop(columns=["seq_out_time_to_event", "seq_out_event"])
y_real = Surv.from_arrays(real["seq_out_event"], real["seq_out_time_to_event"])

X_syn = synthetic.drop(columns=["seq_out_time_to_event", "seq_out_event"])
y_syn = Surv.from_arrays(synthetic["seq_out_event"], synthetic["seq_out_time_to_event"])

In [19]:
X_real_train, X_real_test, y_real_train, y_real_test = train_test_split(
    X_real, y_real, test_size=0.3, random_state=42
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from joblib import dump

from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored
from sksurv.ensemble import RandomSurvivalForest


# =========================================================
# PARAMETRI
# =========================================================

synthetic_fraction = 0.3
n_splits = 
random_state = 42
x = 6000


# =========================================================
# MODEL
# =========================================================

def build_rsf():
    return RandomSurvivalForest(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=15,
        max_features="sqrt",
        n_jobs=1,
        random_state=random_state
    )


# =========================================================
# FIX IMPORTANTISSIMO (alignment pandas/numpy)
# =========================================================

# Assicuriamo coerenza indici
X_syn = X_syn.reset_index(drop=True)
X_real = X_real.reset_index(drop=True)

y_syn = np.array(y_syn)
y_real = np.array(y_real)


# =========================================================
# SAMPLING SINTETICO
# =========================================================

X_syn_sampled = X_syn.sample(
    frac=synthetic_fraction,
    random_state=random_state
)

#y_syn_sampled = y_syn[X_syn_sampled.index.to_numpy()]


# =========================================================
# DATASETS
# =========================================================

datasets = {
    "real_only": (X_real, y_real),

    "real_plus_syn": (
        pd.concat([X_real, X_syn], axis=0).reset_index(drop=True),
        np.concatenate([y_real, y_syn])
    )
}


# =========================================================
# CV SETUP
# =========================================================

kf = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=random_state
)


# =========================================================
# CONTAINERS
# =========================================================

results_cindex = {}
mean_surv_results = {}
mean_chf_results = {}

cmap = cm.get_cmap("viridis", len(datasets))


# =========================================================
# INTERPOLATION UTILITY
# =========================================================

def interpolate_curves(curves, n_points=200):

    max_len = max(len(c) for c in curves)

    common_x = np.linspace(0, max_len - 1, n_points)

    interp = []

    for c in curves:

        old_x = np.linspace(0, len(c) - 1, len(c))

        interp.append(
            np.interp(common_x, old_x, c)
        )

    return common_x, np.mean(interp, axis=0)


# =========================================================
# MAIN LOOP
# =========================================================

for dataset_name, (X_data, y_data) in datasets.items():

    print(f"\nDataset: {dataset_name}")

    cindexes = []
    surv_curves = []
    chf_curves = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_data)):

        print(f"Fold {fold + 1}")

        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]

        y_train = y_data[train_idx]
        y_test = y_data[test_idx]

        # -----------------------------
        # TRAIN
        # -----------------------------
        rsf = build_rsf()
        rsf.fit(X_train, y_train)
        dump(rsf, "/kaggle/working/rsf_model_500.joblib")

        # -----------------------------
        # SURVIVAL
        # -----------------------------
        surv_funcs = rsf.predict_survival_function(X_test)
        surv_values = np.row_stack([fn(fn.x) for fn in surv_funcs])

        surv_curves.append(np.mean(surv_values, axis=0))

        # -----------------------------
        # CUMULATIVE HAZARD
        # -----------------------------
        chf_funcs = rsf.predict_cumulative_hazard_function(X_test)
        chf_values = np.row_stack([fn(fn.x) for fn in chf_funcs])

        chf_curves.append(np.mean(chf_values, axis=0))

        # -----------------------------
        # C-INDEX
        # -----------------------------
        risk_scores = rsf.predict(X_test)

        cindex = concordance_index_censored(
            y_test["event"],
            y_test["time"],
            risk_scores
        )[0]

        cindexes.append(cindex)

    # =====================================================
    # STORE RESULTS
    # =====================================================
    results_cindex[dataset_name] = cindexes

    mean_surv_results[dataset_name] = interpolate_curves(surv_curves)
    mean_chf_results[dataset_name] = interpolate_curves(chf_curves)


# =========================================================
# PLOTS
# =========================================================

# -------------------------
# MEAN SURVIVAL
# -------------------------
plt.figure(figsize=(8, 6))

for i, (name, (t, v)) in enumerate(mean_surv_results.items()):
    plt.step(t, v, where="post", color=cmap(i), label=name)

plt.xlabel("Time")
plt.ylabel("Survival probability")
plt.title(f"Mean Survival Curves - RSF {x} syn")
plt.legend()

plt.savefig(
    f"/kaggle/working/aug_{x}_survival_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# -------------------------
# MEAN CHF
# -------------------------
plt.figure(figsize=(8, 6))

for i, (name, (t, v)) in enumerate(mean_chf_results.items()):
    plt.step(t, v, where="post", color=cmap(i), label=name)

plt.xlabel("Time")
plt.ylabel("Cumulative hazard")
plt.title(f"Mean Cumulative Hazard - RSF {x} syn")
plt.legend()

plt.savefig(
    f"/kaggle/working/aug_{x}_chf_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# -------------------------
# C-INDEX
# -------------------------
plt.figure(figsize=(8, 6))

plt.boxplot(
    results_cindex.values(),
    labels=results_cindex.keys()
)

plt.ylabel("C-index")
plt.title(f"C-index comparison - RSF {x} syn")
plt.grid(False)

plt.savefig(
    f"/kaggle/working/aug_{x}_cindex_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index


# =========================================================
# PARAMETRI
# =========================================================

n_splits = 2
random_state = 42
x = 6000


# =========================================================
# FIX ALIGNMENT
# =========================================================

X_syn = X_syn.reset_index(drop=True)
X_real = X_real.reset_index(drop=True)

y_syn = np.array(y_syn)
y_real = np.array(y_real)


# =========================================================
# DATASET REAL
# =========================================================

df_real = X_real.copy()

df_real["time"] = y_real["time"]
df_real["event"] = y_real["event"].astype(int)


# =========================================================
# DATASET SYN
# =========================================================

df_syn = X_syn.copy()


df_syn["time"] = y_syn["time"]
df_syn["event"] = y_syn["event"].astype(int)


# =========================================================
# DATASETS
# =========================================================

datasets = {

    "real_only": df_real,

    "real_plus_syn": pd.concat(
        [df_real, df_syn],
        axis=0
    ).reset_index(drop=True)
}


# =========================================================
# CV SETUP
# =========================================================

kf = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=random_state
)



# =========================================================
# CONTAINERS
# =========================================================

results_cindex = {}

mean_surv_results = {}
mean_chf_results = {}

cmap = cm.get_cmap(
    "viridis",
    len(datasets)
)


# =========================================================
# INTERPOLATION FUNCTION
# =========================================================

def interpolate_curves(curves, n_points=200):

    max_len = max([
        len(c)
        for c in curves
    ])

    common_x = np.linspace(

        0,
        max_len - 1,
        n_points
    )

    interp = []

    for c in curves:

        old_x = np.linspace(
            0,
            len(c) - 1,
            len(c)
        )

        interp.append(
            np.interp(
                common_x,
                old_x,
                c
            )
        )

    return common_x, np.mean(interp, axis=0)


# =========================================================
# MAIN LOOP
# =========================================================


for dataset_name, df_data in datasets.items():

    print(f"\nDataset: {dataset_name}")

    cindexes = []

    surv_curves = []
    chf_curves = []

    X_data = df_data.drop(
        columns=["time", "event"]
    )

    # -----------------------------------------------------
    # CROSS VALIDATION
    # -----------------------------------------------------

    for fold, (train_idx, test_idx) in enumerate(
        kf.split(X_data)
    ):

        print(f"Fold {fold + 1}")

        train_df = df_data.iloc[train_idx]
        test_df = df_data.iloc[test_idx]

        # -------------------------------------------------
        # TRAIN COX
        # ------------------------------------
-------------

        cph = CoxPHFitter(
            penalizer=0.01
        )

        cph.fit(
            train_df,
            duration_col="time",
            event_col="event"
        )

        # =================================================
        # SURVIVAL FUNCTIONS
        # =================================================

        surv_df = cph.predict_survival_function(
            test_df
        )

        surv_values = surv_df.values.T

        surv_curves.append(
            np.mean(
                surv_values,
                axis=0
            )
        )

        # =================================================

        # CUMULATIVE HAZARD
        # =================================================

        chf_df = cph.predict_cumulative_hazard(
            test_df
        )

        chf_values = chf_df.values.T

        chf_curves.append(
            np.mean(
                chf_values,
                axis=0
            )
        )

        # =================================================
        # C-INDEX
        # =================================================

        partial_hazards = cph.predict_partial_hazard(
            test_df
        )

        cindex = concordance_index(
            test_df["time"],
            -partial_hazards.values.ravel(),

            test_df["event"]
        )

        cindexes.append(cindex)

    # =====================================================
    # SAVE RESULTS
    # =====================================================

    results_cindex[dataset_name] = cindexes

    mean_surv_results[dataset_name] = interpolate_curves(
        surv_curves
    )

    mean_chf_results[dataset_name] = interpolate_curves(
        chf_curves
    )


# =========================================================
# MEAN SURVIVAL CURVES
# =========================================================

plt.figure(figsize=(8, 6))

for i, (name, (t, v)) in enumerate(
    mean_surv_results.items()
):


    plt.step(
        t,
        v,
        where="post",
        color=cmap(i),
        label=name
    )

plt.xlabel("Time")
plt.ylabel("Survival probability")

plt.title(
    f"Mean Survival Curves - CoxPH Lifelines {x} syn"
)

plt.legend()

plt.savefig(
    f"/kaggle/working/lifelines_cox_survival_curve_{x}.png",
    dpi=300,
    bbox_inches="tight"

)

plt.show()


# =========================================================
# MEAN CHF CURVES
# =========================================================

plt.figure(figsize=(8, 6))

for i, (name, (t, v)) in enumerate(
    mean_chf_results.items()
):

    plt.step(
        t,
        v,
        where="post",
        color=cmap(i),
        label=name
    )

plt.xlabel("Time")
plt.ylabel("Cumulative hazard")

plt.title(
    f"Mean Cumulative Hazard - CoxPH Lifelines {x} syn"
)


plt.legend()

plt.savefig(
    f"/kaggle/working/lifelines_cox_chf_curve_{x}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =========================================================
# C-INDEX BOXPLOT
# =========================================================

plt.figure(figsize=(8, 6))

plt.boxplot(
    results_cindex.values(),
    labels=results_cindex.keys()
)

plt.ylabel("C-index")

plt.title(
    f"C-index Comparison - CoxPH Lifelines {x} syn"
)


plt.grid(False)

plt.savefig(
    f"/kaggle/working/lifelines_cox_cindex_{x}_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt

from sksurv.metrics import concordance_index_censored
from sklearn.utils import resample

sizes = ["500", "1000", "1500", "2000", "4000", "6000"]
sizes_plot = ["500", "1k", "1.5k", "2k", "4k", "6k"]

MODEL_DIR = "/kaggle/working/"
N_BOOTSTRAP = 200

cindex_scores = []
ci_lower = []
ci_upper = []

for s in sizes:

    print(f"Model {s}")

    rsf = joblib.load(f"{MODEL_DIR}/rsf_model_{s}.joblib")

    # predict UNA VOLTA
    risk_scores = rsf.predict(X_real_test)

    # c-index full
    cindex = concordance_index_censored(
        y_real_test["event"],
        y_real_test["time"],
        risk_scores
    )[0]

    cindex_scores.append(cindex)

    # bootstrap SOLO sugli indici
    boot_scores = []

    n = len(risk_scores)

    for _ in range(N_BOOTSTRAP):

        idx = np.random.choice(n, n, replace=True)

        score = concordance_index_censored(
            y_real_test["event"][idx],
            y_real_test["time"][idx],
            risk_scores[idx]
        )[0]

        boot_scores.append(score)

    ci_lower.append(np.percentile(boot_scores, 2.5))
    ci_upper.append(np.percentile(boot_scores, 97.5))

# =========================
# PLOT
# =========================

cindex_scores = np.array(cindex_scores)

yerr = np.vstack([
    cindex_scores - np.array(ci_lower),
    np.array(ci_upper) - cindex_scores
])

plt.figure(figsize=(8,5))

plt.errorbar(
    sizes_plot,
    cindex_scores,
    yerr=yerr,
    fmt='o-',
    capsize=5
)

plt.xlabel("Dataset size")
plt.ylabel("C-index")
plt.title("RSF on Test Set")
plt.savefig(
    f"/kaggle/working/aug_summary_rsf_cindex__curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.grid(False)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

# =========================================================
# SURV OBJECTS (come hai già fatto tu)
# =========================================================

X_real = real.drop(columns=["seq_out_time_to_event", "seq_out_event"])
y_real = Surv.from_arrays(
    real["seq_out_event"].astype(bool),
    real["seq_out_time_to_event"]
)

X_syn = synthetic.drop(columns=["seq_out_time_to_event", "seq_out_event"])
y_syn = Surv.from_arrays(
    synthetic["seq_out_event"].astype(bool),
    synthetic["seq_out_time_to_event"]
)

# =========================================================
# PREP DATAFRAME REAL (lifelines format)
# =========================================================

df_real = X_real.copy()
df_real["event"] = y_real["event"]
df_real["time"] = y_real["time"]

# synthetic sorted by seq_id
X_syn = synthetic.copy().sort_values("seq_id").reset_index(drop=True)

# ricostruisco y_syn in modo coerente
y_syn_event = synthetic["seq_out_event"].values
y_syn_time = synthetic["seq_out_time_to_event"].values

# =========================================================
# SETTINGS
# =========================================================

chunks = [500, 1000, 1500, 2000, 4000, 6000]
sizes_plot = ["500", "1k", "1.5k", "2k", "4k", "6k"]

cindex_scores = []
ci_lower = []
ci_upper = []

N_BOOTSTRAP = 100

# =========================================================
# LOOP TRAINING PROGRESSIVO
# =========================================================

for k in chunks:

    print(f"Training CoxPH with real + {k} synthetic")

    # -------------------------
    # build synthetic chunk
    # -------------------------
    syn_part = X_syn.drop(columns=["seq_out_time_to_event", "seq_out_event"]).iloc[:k].copy()

    syn_part["event"] = y_syn_event[:k]
    syn_part["time"] = y_syn_time[:k]

    # -------------------------
    # train set
    # -------------------------
    df_train = pd.concat([df_real, syn_part], axis=0).reset_index(drop=True)

    # -------------------------
    # FIT COX PH
    # -------------------------
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(df_train, duration_col="time", event_col="event")

    # -------------------------
    # PREDICT RISK
    # -------------------------
    risk_scores = cph.predict_partial_hazard(X_real).values.ravel()

    # -------------------------
    # C-INDEX
    # -------------------------
    cindex = concordance_index_censored(
        y_real["event"],
        y_real["time"],
        risk_scores
    )[0]

    cindex_scores.append(cindex)

    # -------------------------
    # BOOTSTRAP
    # -------------------------
    boot_scores = []
    n = len(risk_scores)

    for _ in range(N_BOOTSTRAP):
        idx = np.random.choice(n, n, replace=True)

        score = concordance_index_censored(
            y_real["event"][idx],
            y_real["time"][idx],
            risk_scores[idx]
        )[0]

        boot_scores.append(score)

    ci_lower.append(np.percentile(boot_scores, 2.5))
    ci_upper.append(np.percentile(boot_scores, 97.5))

# =========================================================
# PLOT
# =========================================================

cindex_scores = np.array(cindex_scores)

yerr = np.vstack([
    cindex_scores - np.array(ci_lower),
    np.array(ci_upper) - cindex_scores
])

plt.figure(figsize=(8,5))

plt.errorbar(
    sizes_plot,
    cindex_scores,
    yerr=yerr,
    fmt='o-',
    capsize=5
)

plt.xlabel("Synthetic samples added")
plt.ylabel("C-index")
plt.title("CoxPH (lifelines) Test Set")
plt.savefig(
    f"/kaggle/working/aug_summary_coxph_cindex__curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.grid(False)
plt.show()